## **Build an AI Math Assistant with LangChain Tool Calling**

### import libraries

In [4]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import torch

In [5]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

In [6]:
device = 0 if torch.cuda.is_available() else -1

In [16]:
def load_llm(model_name=MODEL_NAME):
    
    try:
        if not model_name:
            raise ValueError("Model name is empty or none")
        

        pipeline = HuggingFacePipeline.from_model_id(
            model_id=model_name,
            task="text-generation",
            device=device,
            pipeline_kwargs={
                "temperature": 0.7,
                "max_new_tokens": 20,
                "do_sample": True,  # needed for temperature to actually take effect
            }
        ) 
        
        llm_model = ChatHuggingFace(llm=pipeline)
        
        return llm_model
        
    except ValueError as e:
        print(f"Value error; {e}")
        raise
        
    except Exception as e:
        print(f"Error in load llm: {e}")
        raise

In [17]:
llm_model = load_llm()

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2255.88it/s]


In [18]:
llm_model.invoke("Hi how are you?")

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


AIMessage(content="<s>[INST] Hi how are you?[/INST]Hello! I'm an AI, so I don't have feelings, but I'm", additional_kwargs={}, response_metadata={}, id='run--019f3afd-4c83-7230-a4e8-6fcae47ac885-0')

Method 01:

Tool Calling with Langchain agent

In [ ]:
def add_numbers(input:str)-> dict:
    
    """
    Add list of numbers provided in list of numbers or exracted from a string given and return their sum as
    {"result": total}.

    """
    
    numbers = [int(x) for x in input.replace(",","").split() if x.isdigit()]
    return {
        "result": sum(numbers)
    }

In [20]:
from langchain.agents import Tool

In [23]:
add_numbers_tool = Tool(
    name="AddTool",
    func=add_numbers,
    description="Adds a list of numbers and returns the result."
)

print("AddTool object: ", add_numbers_tool)

AddTool object:  name='AddTool' description='Adds a list of numbers and returns the result.' func=Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x000001A63270B1A0>)


In [26]:
print("Tool name:")
print(add_numbers_tool.name)

print("\nTool description:")
print(add_numbers_tool.description)

print("\nTool function")
print(add_numbers_tool.invoke)

Tool name:
AddTool

Tool description:
Adds a list of numbers and returns the result.

Tool function
<bound method BaseTool.invoke of Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=Tool(name='AddTool', description='Adds a list of numbers and returns the result.', func=<function add_numbers at 0x000001A63270B1A0>))>
